# DamageACT-U G3 — Final Devil-Audited Frozen T4 Parity Notebook

This notebook preserves the frozen G3 checkpoint-parity experiment and hardens only the Kaggle transport/wrapper validation around it.

It prefers the byte-identical preserved execution archive when available, verifies the original archive size/SHA-256, CRC, all 14 payload hashes, frozen runner/replay metadata, exact T4 environment, and the xBD root. It then runs the unchanged hash-verified frozen runner.

After the runner finishes, the wrapper independently requires: return code 0; `POST 96/96`; `Siamese 96/96`; all four frozen numerical comparison summaries with `allclose=True`; exactly one `STRICT_PARITY_PASS: True`; the exact expected result-ZIP path; a non-empty ZIP; and a clean ZIP CRC.

No new scientific threshold is introduced. The frozen `atol=1e-5` and `rtol=1e-5`, checkpoints, replay batches, selector, source, and model logic are unchanged.


In [1]:
# ============================================================
# DamageACT-U G3 — FINAL FROZEN T4 CHECKPOINT PARITY
# Corrected Kaggle one-cell notebook
# ============================================================

from pathlib import Path
import sys
import os
import json
import hashlib
import shutil
import zipfile
import subprocess
import platform
import re
import math


# ============================================================
# FROZEN VALUES — DO NOT CHANGE
# ============================================================

EXPECTED_EXECUTION_ZIP_SHA = (
    "289c5753051af703fa58baf37785d47269d058a1cf31217a2a299acfb15ea519"
)

EXPECTED_EXECUTION_ZIP_BYTES = 159185660

EXPECTED_RUNNER_SHA = (
    "cda8f058123ecbe9f63e65ded32fc8ba8161654e07a0e769980f3ecb9fdc0ea9"
)

EXPECTED_RUNNER_MANIFEST_SHA = (
    "24f93352d9faa35b0222e5f167eb92efdb4b4943170899feba39206b4077f953"
)

EXPECTED_REPLAY_LOCK_SHA = (
    "78f6a5eda2aed875f7b65f08879c7ce85987766c9283e10f6ad7fc6a7547bef0"
)

EXPECTED_REPLAY_COMMIT = "847da83"

ATOL = 1e-5
RTOL = 1e-5

# Frozen per-file identities from the committed G3 execution bundle.
EXPECTED_PAYLOAD_HASHES = {'audit/g3_canonical_batch_replay_lock.json': '78f6a5eda2aed875f7b65f08879c7ce85987766c9283e10f6ad7fc6a7547bef0', 'audit/g3_clean_parity_panel_23.csv': '2924bc88beeda3507c8cf2178db53c2d17eca7483c483a61ca59087a87636c43', 'audit/g3_clean_parity_selector_96.csv': '131ede6d2c4bf3efbd0e62a219994bc8228457aeaeb6022e5068d1586ee97f03', 'audit/g3_parity_execution_protocol.json': '112ade4ac863dd7bc7df03a6c2544be6b3757ca0034f864556738851b2d2409c', 'audit/g3_parity_runner_manifest.json': '24f93352d9faa35b0222e5f167eb92efdb4b4943170899feba39206b4077f953', 'audit/g3_parity_selection_lock.json': 'cf463ca5d680ba7ede33d498350692e2782f5d716bc527629dd07a77e210883c', 'checkpoints/experts/post_seed42_best_state_dict.pt': 'df6b607973ea74e1a6aa7eb640459b65c7a4e1bdaabc7fce956d2c757374967c', 'checkpoints/experts/siamese_seed42_best_state_dict.pt': '3d6466108cf44e994c0c7b7bb1745e549807d89601e45898911de34a455d73a5', 'manifests/buildings/event_test_buildings_rebuilt.csv.gz': '1dda199c0faa444e955ecf7b131872dc5c5a713482914e821cd916aea4b37a2a', 'predictions/heldout_events/event_test_clean_expert_predictions.csv.gz': 'f0657fc72629acb8730464897c44040cde65942966a81f47c956f3c5b1bb09f2', 'scripts/g3_checkpoint_parity_t4.py': 'cda8f058123ecbe9f63e65ded32fc8ba8161654e07a0e769980f3ecb9fdc0ea9', 'vendor/frozen_phase7d_source/src/data/crop_utils.py': '8c34139a0f96eaf8246dd228cf27c83386d6c1ca049426c25d28666576589593', 'vendor/frozen_phase7d_source/src/data/dataset.py': '165a71c4ebf6adea14f20f735bc487e883c5448e5ccfaf4fe8b74e49a3a77044', 'vendor/frozen_phase7d_source/src/models/damage_models.py': '9f9ee3382902f339309ddf03a203b882d42e8674de7bf8ee3d18b14863478f2d'}


# ============================================================
# OPTIONAL
#
# Normally leave this as None.
# Only fill it if automatic xBD detection says AMBIGUOUS.
#
# Example:
# XBD_ROOT_OVERRIDE = "/kaggle/input/damageactu-xbd-private"
# ============================================================

XBD_ROOT_OVERRIDE = None


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


INPUT_ROOT = Path("/kaggle/input")

WORKING_BUNDLE_ROOT = Path(
    "/kaggle/working/DamageACTU_G3_Parity_Execution_Bundle"
)

RESULT_ZIP = Path(
    "/kaggle/working/DamageACTU_G3_Parity_Results.zip"
)


print("=" * 70)
print("DamageACT-U G3 Frozen Checkpoint Parity")
print("=" * 70)


# ============================================================
# 1. ENVIRONMENT
# ============================================================

import torch
import torchvision
import PIL


python_version = platform.python_version()

torch_version = torch.__version__

torchvision_version = torchvision.__version__

cuda_version = torch.version.cuda

gpu_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "NO GPU"
)


print("\n[1] ENVIRONMENT")

print("Python      :", python_version)
print("Torch       :", torch_version)
print("Torchvision :", torchvision_version)
print("CUDA        :", cuda_version)
print("GPU         :", gpu_name)
print("Pillow      :", PIL.__version__)


environment_errors = []


if not python_version.startswith("3.12."):

    environment_errors.append(
        f"Python must be 3.12.x; got {python_version}"
    )


if torch_version != "2.10.0+cu128":

    environment_errors.append(
        "Torch must be 2.10.0+cu128; "
        f"got {torch_version}"
    )


if torchvision_version != "0.25.0+cu128":

    environment_errors.append(
        "Torchvision must be 0.25.0+cu128; "
        f"got {torchvision_version}"
    )


if cuda_version != "12.8":

    environment_errors.append(
        f"CUDA must be 12.8; got {cuda_version}"
    )


if "T4" not in gpu_name:

    environment_errors.append(
        f"GPU must be T4; got {gpu_name}"
    )


if environment_errors:

    print("\nENVIRONMENT GATE: FAIL")

    for error in environment_errors:

        print(" -", error)

    raise RuntimeError(
        "\nSTOP HERE.\n"
        "Do not install/change packages yet. "
        "Send this environment output to ChatGPT."
    )


print("ENVIRONMENT GATE: PASS")


# ============================================================
# 2. SHOW ATTACHED KAGGLE INPUTS
# ============================================================

print("\n[2] ATTACHED INPUTS")


assert INPUT_ROOT.is_dir(), (
    "/kaggle/input does not exist."
)


for p in sorted(INPUT_ROOT.iterdir()):

    print(" -", p)


# ============================================================
# 3. LOCATE THE FROZEN EXECUTION BUNDLE
#
# IMPORTANT:
# Kaggle may expose a dataset either as the original archive or
# as an already-extracted directory tree. The scientific identity
# is the frozen payload bytes, not the outer ZIP container.
#
# Therefore this notebook accepts:
#   A) the exact frozen archive (if present), OR
#   B) the auto-extracted bundle, but ONLY after its frozen
#      SHA256SUMS content and control-file identities match.
#
# No tolerance, checkpoint, replay, or model setting is changed.
# ============================================================

print("\n[3] LOCATING FROZEN EXECUTION BUNDLE")

# Kaggle layouts supported:
#   /kaggle/input/datasets/<owner>/<dataset-slug>/
#   /kaggle/input/<dataset-slug>/

dataset_roots = []

datasets_root = Path("/kaggle/input/datasets")
if datasets_root.is_dir():
    for owner_dir in datasets_root.iterdir():
        if not owner_dir.is_dir():
            continue
        try:
            for dataset_dir in owner_dir.iterdir():
                if dataset_dir.is_dir():
                    dataset_roots.append(dataset_dir)
        except (PermissionError, OSError):
            pass

for p in Path("/kaggle/input").iterdir():
    if p.is_dir() and p.name != "datasets":
        dataset_roots.append(p)

dataset_roots = list(dict.fromkeys(dataset_roots))

print("Attached dataset roots =", len(dataset_roots))
for root in dataset_roots:
    print(" -", root)

# ------------------------------------------------------------
# A. Inspect auto-extracted execution-bundle candidates.
# IMPORTANT: a candidate is accepted only when ALL 14 payload files
# are physically present and byte-identical to the frozen manifest.
# This prevents the Kaggle transport issue seen in the previous run,
# where two nested .csv.gz payloads were silently omitted.
# ------------------------------------------------------------

def bundle_signature_ok(root):
    root = Path(root)
    needed = [
        root / "SHA256SUMS.json",
        root / "scripts/g3_checkpoint_parity_t4.py",
        root / "audit/g3_parity_runner_manifest.json",
        root / "audit/g3_canonical_batch_replay_lock.json",
    ]
    return all(p.is_file() for p in needed)


def bundle_manifest_matches_frozen(root):
    root = Path(root)
    sums_path = root / "SHA256SUMS.json"
    try:
        sums = json.loads(sums_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return (
        isinstance(sums, dict)
        and len(sums) == 14
        and sums == EXPECTED_PAYLOAD_HASHES
    )


def verify_frozen_payload_tree(root):
    """Return (missing, mismatched) for the frozen 14-file payload tree."""
    root = Path(root)
    missing = []
    mismatched = []

    for relative_path, expected_sha in EXPECTED_PAYLOAD_HASHES.items():
        p = root / relative_path
        if not p.is_file():
            missing.append(relative_path)
            continue

        try:
            actual_sha = sha256_file(p)
        except OSError as exc:
            mismatched.append((relative_path, expected_sha, f"READ_ERROR:{exc}"))
            continue

        if actual_sha != expected_sha:
            mismatched.append((relative_path, expected_sha, actual_sha))

    return missing, mismatched


candidate_bundle_roots = []
for root in dataset_roots:
    candidate_bundle_roots.append(root)

    # One shallow wrapper level is enough for uploaded bundle layouts.
    try:
        for child in root.iterdir():
            if child.is_dir() and child.name.lower() not in {
                "train", "tier1", "tier3", "test", "hold", "holdout",
                "images", "labels", "xbd_full"
            }:
                candidate_bundle_roots.append(child)
    except (PermissionError, OSError):
        pass

candidate_bundle_roots = list(dict.fromkeys(candidate_bundle_roots))

complete_extracted_roots = []
incomplete_frozen_roots = []

for root in candidate_bundle_roots:
    if not bundle_signature_ok(root):
        continue

    print("\nExtracted-bundle signature found:", root)

    if not bundle_manifest_matches_frozen(root):
        print("  SHA256SUMS content: DOES NOT MATCH FROZEN 14-FILE MANIFEST")
        continue

    runner = root / "scripts/g3_checkpoint_parity_t4.py"
    runner_manifest = root / "audit/g3_parity_runner_manifest.json"
    replay_lock = root / "audit/g3_canonical_batch_replay_lock.json"

    try:
        runner_ok = sha256_file(runner) == EXPECTED_RUNNER_SHA
        runner_manifest_ok = (
            sha256_file(runner_manifest) == EXPECTED_RUNNER_MANIFEST_SHA
        )
        replay_ok = sha256_file(replay_lock) == EXPECTED_REPLAY_LOCK_SHA
    except OSError:
        continue

    print("  runner identity        =", runner_ok)
    print("  runner manifest        =", runner_manifest_ok)
    print("  replay lock            =", replay_ok)

    if not (runner_ok and runner_manifest_ok and replay_ok):
        continue

    missing_now, mismatched_now = verify_frozen_payload_tree(root)

    if not missing_now and not mismatched_now:
        print("  all 14 payload files   = PRESENT + HASH VERIFIED")
        complete_extracted_roots.append(root)
    else:
        print("  complete payload tree  = NO")
        if missing_now:
            print("  missing:")
            for rel in missing_now:
                print("   -", rel)
        if mismatched_now:
            print("  hash mismatches:")
            for item in mismatched_now:
                print("   -", item[0])
        incomplete_frozen_roots.append(
            (root, missing_now, mismatched_now)
        )

complete_extracted_roots = list(dict.fromkeys(complete_extracted_roots))


# ------------------------------------------------------------
# B. Exact-archive fallback.
#
# Kaggle may auto-extract an uploaded ZIP and omit nested .csv.gz
# members. To preserve the original bytes, attach a Kaggle dataset
# containing an UNMODIFIED COPY of the original execution ZIP renamed
# to any non-archive extension (recommended: .bin).
#
# Required exact archive identity:
#   bytes  = 159185660
#   sha256 = 289c5753051af703fa58baf37785d47269d058a1cf31217a2a299acfb15ea519
#
# The filename itself is irrelevant; size + SHA-256 are authoritative.
# ------------------------------------------------------------

archive_candidates = []


def candidate_transport_files(dataset_root):
    """
    Enumerate only shallow files so we never recursively scan raw xBD.
    A preserved .bin archive should normally be at dataset-root level.
    """
    dataset_root = Path(dataset_root)
    out = []

    try:
        for p in dataset_root.iterdir():
            if p.is_file():
                out.append(p)
            elif p.is_dir() and p.name.lower() not in {
                "train", "tier1", "tier3", "test", "hold", "holdout",
                "images", "labels", "xbd_full"
            }:
                try:
                    for q in p.iterdir():
                        if q.is_file():
                            out.append(q)
                except (PermissionError, OSError):
                    pass
    except (PermissionError, OSError):
        pass

    return out


for root in dataset_roots:
    for p in candidate_transport_files(root):
        try:
            if p.stat().st_size != EXPECTED_EXECUTION_ZIP_BYTES:
                continue
            got = sha256_file(p)
        except OSError:
            continue

        if got == EXPECTED_EXECUTION_ZIP_SHA:
            archive_candidates.append(p)

archive_candidates = list(dict.fromkeys(archive_candidates))

if archive_candidates:
    print("\nExact preserved archive candidate(s):")
    for p in archive_candidates:
        print(" -", p)


# ------------------------------------------------------------
# C. Select exactly one authoritative representation.
# ------------------------------------------------------------

# Prefer the byte-identical preserved archive whenever available.
# This is the strongest provenance route because the outer container
# identity (size + SHA-256) is also frozen. A complete extracted tree
# remains an allowed fallback only when no exact preserved archive is attached.

if len(archive_candidates) > 1:
    print("\nMultiple byte-identical preserved archive copies:")
    for p in archive_candidates:
        print(" -", p)
    raise RuntimeError(
        "More than one exact frozen execution archive is attached. "
        "Keep only one preserved archive copy."
    )

if len(complete_extracted_roots) > 1:
    print("\nMultiple complete byte-verified extracted bundle roots:")
    for p in complete_extracted_roots:
        print(" -", p)
    raise RuntimeError(
        "More than one complete byte-verified G3 bundle was found. "
        "Attach only one authoritative execution-bundle dataset."
    )

if len(archive_candidates) == 1:
    frozen_archive = archive_candidates[0]
    print("\nUsing exact byte-preserved frozen archive:", frozen_archive)

    if frozen_archive.stat().st_size != EXPECTED_EXECUTION_ZIP_BYTES:
        raise RuntimeError("Frozen archive byte size changed.")

    if sha256_file(frozen_archive) != EXPECTED_EXECUTION_ZIP_SHA:
        raise RuntimeError("Frozen archive SHA-256 changed.")

    shutil.rmtree(WORKING_BUNDLE_ROOT, ignore_errors=True)
    WORKING_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(frozen_archive, "r") as z:
        names = z.namelist()

        if len(names) != 15:
            raise RuntimeError(
                f"Frozen archive should contain exactly 15 files; found {len(names)}"
            )

        bad_member = z.testzip()

        if bad_member is not None:
            raise RuntimeError(f"ZIP CRC failure at: {bad_member}")

        z.extractall(WORKING_BUNDLE_ROOT)

    missing_after, mismatched_after = verify_frozen_payload_tree(
        WORKING_BUNDLE_ROOT
    )

    if missing_after or mismatched_after:
        print("\nPost-extraction verification failed.")

        if missing_after:
            print("Missing after extraction:")
            for rel in missing_after:
                print(" -", rel)

        if mismatched_after:
            print("Hash mismatch after extraction:")
            for item in mismatched_after:
                print(" -", item)

        raise RuntimeError(
            "Exact frozen archive extracted, but payload verification failed."
        )

    bundle_root = WORKING_BUNDLE_ROOT
    bundle_transport_mode = "BYTE_PRESERVED_ARCHIVE_EXTRACTED_ALL_14_VERIFIED"

    print("FROZEN ARCHIVE BYTE IDENTITY: PASS")
    print("FROZEN ARCHIVE CRC: PASS")
    print("ALL 14 EXTRACTED PAYLOAD HASHES: PASS")
    print("BUNDLE ROOT =", bundle_root)

elif len(complete_extracted_roots) == 1:
    bundle_root = complete_extracted_roots[0]
    bundle_transport_mode = "KAGGLE_EXTRACTED_ALL_14_HASH_VERIFIED"

    print(
        "\nNo exact preserved archive was attached, but one complete "
        "14/14 byte-verified extracted execution bundle is available."
    )
    print("BUNDLE ROOT =", bundle_root)

else:
    print("\nSTOP: no scientifically complete G3 bundle is available.")

    if incomplete_frozen_roots:
        print(
            "\nThe mounted execution-bundle dataset has the correct frozen "
            "manifest/control files, but Kaggle omitted payload bytes."
        )

        for root, missing_now, mismatched_now in incomplete_frozen_roots:
            print("\nIncomplete mounted root:", root)

            if missing_now:
                print("Missing payloads:")
                for rel in missing_now:
                    print(" -", rel)

            if mismatched_now:
                print("Mismatched payloads:")
                for item in mismatched_now:
                    print(" -", item[0])

    print(
        "\nRequired repair:\n"
        "Attach the byte-preserved copy of the original frozen execution ZIP "
        "(recommended extension: .bin).\n"
        f"Expected bytes  = {EXPECTED_EXECUTION_ZIP_BYTES}\n"
        f"Expected SHA256 = {EXPECTED_EXECUTION_ZIP_SHA}\n"
        "Do NOT recompress the ZIP, regenerate missing CSVs, or change atol/rtol."
    )

    raise RuntimeError(
        "BYTE-PRESERVED EXECUTION ARCHIVE REQUIRED: "
        "no complete frozen payload representation is available."
    )

print("\nBUNDLE TRANSPORT MODE =", bundle_transport_mode)

# ============================================================
# 4. VERIFY ALL 14 FROZEN PAYLOAD FILES
# ============================================================

print(
    "\n[4] VERIFYING COMPLETE FROZEN BUNDLE"
)


runner_path = (
    bundle_root
    /
    "scripts/g3_checkpoint_parity_t4.py"
)


runner_manifest_path = (
    bundle_root
    /
    "audit/g3_parity_runner_manifest.json"
)


replay_lock_path = (
    bundle_root
    /
    "audit/g3_canonical_batch_replay_lock.json"
)


sums_path = (
    bundle_root
    /
    "SHA256SUMS.json"
)


required_control_files = [

    runner_path,

    runner_manifest_path,

    replay_lock_path,

    sums_path,

]


for p in required_control_files:

    assert p.is_file(), (
        f"Required frozen control file missing: {p}"
    )


# ------------------------------------------------------------
# Known identity anchors
# ------------------------------------------------------------

runner_sha = sha256_file(
    runner_path
)


runner_manifest_sha = sha256_file(
    runner_manifest_path
)


replay_lock_sha = sha256_file(
    replay_lock_path
)


print(
    "RUNNER SHA256          =",
    runner_sha,
)

print(
    "RUNNER MANIFEST SHA256 =",
    runner_manifest_sha,
)

print(
    "REPLAY LOCK SHA256     =",
    replay_lock_sha,
)


assert (
    runner_sha
    ==
    EXPECTED_RUNNER_SHA
)


assert (
    runner_manifest_sha
    ==
    EXPECTED_RUNNER_MANIFEST_SHA
)


assert (
    replay_lock_sha
    ==
    EXPECTED_REPLAY_LOCK_SHA
)


# ------------------------------------------------------------
# Verify every one of the 14 payload hashes
# ------------------------------------------------------------

sums = json.loads(
    sums_path.read_text(
        encoding="utf-8"
    )
)

assert sums == EXPECTED_PAYLOAD_HASHES, (
    "SHA256SUMS.json does not equal the frozen 14-file manifest."
)

print(
    "HASHED PAYLOAD FILES =",
    len(sums),
)


assert len(sums) == 14, (
    "Expected exactly 14 frozen payload files; "
    f"found {len(sums)}"
)


missing = []

mismatched = []


for relative_path, expected_sha in sums.items():

    p = (
        bundle_root
        /
        relative_path
    )


    if not p.is_file():

        missing.append(
            relative_path
        )

        continue


    actual_sha = sha256_file(
        p
    )


    if actual_sha != expected_sha:

        mismatched.append(
            (
                relative_path,
                expected_sha,
                actual_sha,
            )
        )


if missing:

    print(
        "\nMISSING FILES:"
    )

    for p in missing:

        print(
            " -",
            p,
        )


if mismatched:

    print(
        "\nHASH MISMATCHES:"
    )

    for item in mismatched:

        print(
            item
        )


assert not missing, (
    "Frozen bundle still has missing payload files."
)


assert not mismatched, (
    "Frozen bundle contains payload hash mismatches."
)


# ------------------------------------------------------------
# Validate runner freeze metadata
# ------------------------------------------------------------

runner_manifest = json.loads(
    runner_manifest_path.read_text(
        encoding="utf-8"
    )
)


assert (
    runner_manifest[
        "status"
    ]
    ==
    "PRE_INFERENCE_RUNNER_FROZEN_VALIDATED"
)


assert (
    runner_manifest[
        "runner_sha256"
    ]
    ==
    EXPECTED_RUNNER_SHA
)


assert (
    runner_manifest[
        "canonical_batch_replay_commit"
    ]
    ==
    EXPECTED_REPLAY_COMMIT
)


assert (
    runner_manifest[
        "predeclared_atol"
    ]
    ==
    ATOL
)


assert (
    runner_manifest[
        "predeclared_rtol"
    ]
    ==
    RTOL
)


assert (
    runner_manifest[
        "contains_regenerated_outputs"
    ]
    is False
)


assert (
    runner_manifest[
        "runner_has_training_code"
    ]
    is False
)


assert (
    runner_manifest[
        "runner_has_tuning_code"
    ]
    is False
)


print(
    "\nFINAL EXECUTION BUNDLE IDENTITY: PASS"
)

print(
    "ALL 14 BUNDLE PAYLOAD HASHES: PASS"
)

print(
    "\nBUNDLE TRANSPORT/INTEGRITY STAGE: COMPLETE"
)
# ============================================================
# 5. FIND RAW xBD ROOT
# ============================================================

print("\n[5] LOCATING xBD DATASET")


SUBSET_NAMES = (
    "train",
    "tier1",
    "tier3",
    "test",
    "hold",
    "holdout",
)


def recognized_subsets(root):

    root = Path(root)

    result = []

    for subset in SUBSET_NAMES:

        q = root / subset

        if (
            (q / "images").is_dir()
            and
            (q / "labels").is_dir()
        ):

            result.append(subset)

    return result


def iter_candidate_xbd_roots(dataset_root, max_wrapper_depth=2):
    """
    Yield only shallow directory candidates beneath one attached Kaggle
    dataset root. Never recurse into xBD subset/image/label trees.

    This supports both:
        <dataset-root>/train, tier3, test, ...
    and wrapped layouts such as:
        <dataset-root>/xbd_full/train, tier3, test, ...
    """

    dataset_root = Path(dataset_root)

    if not dataset_root.is_dir():

        return

    queue = [
        (dataset_root, 0)
    ]

    visited = set()

    stop_names = set(SUBSET_NAMES) | {
        "images",
        "labels",
    }

    while queue:

        current, depth = queue.pop(0)

        try:

            key = str(
                current.resolve()
            )

        except OSError:

            key = str(current)


        if key in visited:

            continue

        visited.add(key)

        yield current


        if depth >= max_wrapper_depth:

            continue


        try:

            children = list(
                current.iterdir()
            )

        except (
            PermissionError,
            OSError,
        ):

            continue


        for child in children:

            try:

                is_directory = child.is_dir()

            except OSError:

                continue


            if not is_directory:

                continue


            if child.name.lower() in stop_names:

                # Never descend into xBD partitions or the huge payload dirs.
                continue


            queue.append(
                (
                    child,
                    depth + 1,
                )
            )


if XBD_ROOT_OVERRIDE is not None:

    XBD_ROOT = Path(
        XBD_ROOT_OVERRIDE
    )


    assert XBD_ROOT.is_dir(), (
        f"Manual XBD root not found: {XBD_ROOT}"
    )


    found_subsets = recognized_subsets(
        XBD_ROOT
    )


    assert len(found_subsets) >= 3, (
        "Manual path does not look like "
        "the xBD dataset root."
    )


else:

    candidates = []

    seen_candidates = set()


    # Search from the dataset roots already discovered in section 3.
    # The previous notebook searched only down to the dataset directory
    # itself. Kaggle mounted this xBD copy one wrapper deeper:
    #
    #   .../damageactu-xbd-full/xbd_full/<subset>/...
    #
    # so the real root was never examined. This shallow wrapper search
    # fixes that path bug without walking images/ or labels/.

    for dataset_root in dataset_roots:

        for root in iter_candidate_xbd_roots(
            dataset_root,
            max_wrapper_depth=2,
        ):

            found = recognized_subsets(
                root
            )


            if len(found) < 3:

                continue


            try:

                root_key = str(
                    root.resolve()
                )

            except OSError:

                root_key = str(root)


            if root_key in seen_candidates:

                continue


            seen_candidates.add(
                root_key
            )

            candidates.append(
                (
                    root,
                    found,
                )
            )


    print(
        "xBD candidates:"
    )


    for i, (
        root,
        subsets,
    ) in enumerate(
        candidates
    ):

        print(
            f"[{i}] {root}"
        )

        print(
            "    subsets:",
            subsets,
        )


    if len(candidates) == 0:

        print(
            "\nNo xBD root was recognized."
        )

        print(
            "Checked attached dataset roots and up to "
            "two wrapper-directory levels beneath each."
        )

        print(
            "A valid xBD root must contain at least three "
            "recognized partitions, each with both images/ "
            "and labels/ directories."
        )

        raise RuntimeError(
            "\nNo xBD root detected.\n"
            "The raw xBD dataset appears to be missing or "
            "mounted in an unexpected layout.\n"
            "If needed, set XBD_ROOT_OVERRIDE to the directory "
            "that directly contains train/tier3/test/hold."
        )


    best_score = max(
        len(subsets)
        for _, subsets
        in candidates
    )


    best = [
        (
            root,
            subsets,
        )
        for root, subsets
        in candidates
        if len(subsets)
        ==
        best_score
    ]


    if len(best) != 1:

        print(
            "\nAMBIGUOUS xBD ROOT."
        )

        for root, subsets in best:

            print(
                " -",
                root,
                "subsets =",
                subsets,
            )

        print(
            "Set XBD_ROOT_OVERRIDE at the "
            "top of this cell to the correct path."
        )


        raise RuntimeError(
            "Multiple equally complete xBD roots detected."
        )


    XBD_ROOT, found_subsets = (
        best[0]
    )


print(
    "\nXBD_ROOT =",
    XBD_ROOT,
)

print(
    "Subsets =",
    found_subsets,
)

print(
    "XBD ROOT CHECK: PASS"
)


# ============================================================
# 6. CLEAN ONLY OLD G3 OUTPUTS
# ============================================================

print("\n[6] CLEANING PREVIOUS G3 OUTPUTS")


possible_output_dirs = [

    Path(
        "/kaggle/working/"
        "damageactu_g3_checkpoint_parity"
    ),

    Path(
        "/kaggle/working/"
        "damageactu_g3_stage"
    ),

]


for p in possible_output_dirs:

    if p.exists():

        shutil.rmtree(
            p
        )


old_result_zip = Path(
    "/kaggle/working/"
    "DamageACTU_G3_Parity_Results.zip"
)


if old_result_zip.exists():

    old_result_zip.unlink()


print(
    "Previous G3 working outputs cleared."
)


# ============================================================
# 7. RUN FROZEN PARITY AUDIT
#
# THIS RUN REPLAYS THE FROZEN MODEL INFERENCE; NO TRAINING OR TUNING.
# ============================================================

print("\n[7] STARTING FROZEN T4 PARITY RUN")


cmd = [

    sys.executable,

    str(
        runner_path
    ),

    "--bundle-root",

    str(
        bundle_root
    ),

    "--xbd-root",

    str(
        XBD_ROOT
    ),

]


print(
    "Command:"
)

print(
    " ".join(
        cmd
    )
)


print(
    "\nReplaying:"
)

print(
    "  78 historical batches"
)

print(
    "  7445 canonical rows"
)

print(
    "  180 scenes"
)

print(
    "  96 frozen comparison targets"
)

print(
    "  final historical batch size = 53"
)

print(
    "\nDO NOT INTERRUPT UNLESS AN ERROR APPEARS.\n"
)


# Run the exact frozen, hash-verified runner while preserving its stdout.
# The output is streamed live to the notebook AND retained for strict
# machine-readable verification below.
runner_output_lines = []

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for output_line in process.stdout:
    print(output_line, end="")
    runner_output_lines.append(output_line)

runner_return_code = process.wait()
runner_stdout = "".join(runner_output_lines)

runner_stdout_path = Path("/kaggle/working/DamageACTU_G3_Parity_Runner_stdout.txt")
runner_stdout_path.write_text(runner_stdout, encoding="utf-8")
print("RUNNER STDOUT SAVED =", runner_stdout_path)

print(
    "\nRUNNER RETURN CODE =",
    runner_return_code,
)

if runner_return_code != 0:
    raise RuntimeError(
        "\nFROZEN PARITY RUN FAILED.\n"
        "Do NOT change tolerances/settings.\n"
        "The exact frozen runner returned a non-zero exit code."
    )

print(
    "FROZEN RUNNER COMPLETED: PASS"
)


# ============================================================
# 8. VERIFY AUTHORITATIVE FROZEN-RUNNER RESULT
# ============================================================

print("\n[8] VERIFYING AUTHORITATIVE FROZEN-RUNNER RESULT")

# The frozen runner itself is the authoritative producer of the parity verdict.
# Previous notebook versions incorrectly required a separate standalone JSON
# containing a top-level STRICT_PARITY_PASS key. The frozen runner does not
# create such a JSON in /kaggle/working, so that wrapper check produced a false
# post-processing failure even when the runner had already passed.
#
# Here we instead parse the exact stdout emitted by the byte/hash-verified
# frozen runner. We require unique, exact-format evidence and cross-check the
# frozen target cardinalities.

strict_matches = re.findall(
    r"(?m)^STRICT_PARITY_PASS:\s*(True|False)\s*$",
    runner_stdout,
)

if len(strict_matches) != 1:
    raise RuntimeError(
        "Expected exactly one authoritative STRICT_PARITY_PASS line "
        f"from the frozen runner; found {len(strict_matches)}"
    )

strict_pass = strict_matches[0] == "True"

post_matches = re.findall(
    r"(?m)^POST prediction matches\s*:\s*(\d+)/(\d+)\s*$",
    runner_stdout,
)

siamese_matches = re.findall(
    r"(?m)^Siamese prediction matches\s*:\s*(\d+)/(\d+)\s*$",
    runner_stdout,
)

if len(post_matches) != 1:
    raise RuntimeError(
        "Expected exactly one POST prediction-match summary from the "
        f"frozen runner; found {len(post_matches)}"
    )

if len(siamese_matches) != 1:
    raise RuntimeError(
        "Expected exactly one Siamese prediction-match summary from the "
        f"frozen runner; found {len(siamese_matches)}"
    )

post_num, post_den = map(int, post_matches[0])
siamese_num, siamese_den = map(int, siamese_matches[0])

if (post_num, post_den) != (96, 96):
    raise AssertionError(
        f"Frozen POST parity cardinality changed: {post_num}/{post_den}"
    )

if (siamese_num, siamese_den) != (96, 96):
    raise AssertionError(
        f"Frozen Siamese parity cardinality changed: "
        f"{siamese_num}/{siamese_den}"
    )

comparison_pattern = re.compile(
    r"(?m)^(post_logits|post_probabilities|siamese_logits|siamese_probabilities)"
    r"\s+max_abs=\s*([0-9eE+\-.]+)"
    r"\s+mean_abs=\s*([0-9eE+\-.]+)"
    r"\s+median_abs=\s*([0-9eE+\-.]+)"
    r"\s+allclose=\s*(True|False)\s*$"
)

comparison_matches = comparison_pattern.findall(runner_stdout)

if len(comparison_matches) != 4:
    raise RuntimeError(
        "Expected exactly four frozen numerical-comparison summaries; "
        f"found {len(comparison_matches)}"
    )

expected_comparison_names = {
    "post_logits",
    "post_probabilities",
    "siamese_logits",
    "siamese_probabilities",
}

observed_comparison_names = {
    row[0]
    for row in comparison_matches
}

if observed_comparison_names != expected_comparison_names:
    raise RuntimeError(
        "Frozen numerical-comparison summary names changed: "
        f"{sorted(observed_comparison_names)}"
    )

comparison_stats = {}

for name, max_abs_s, mean_abs_s, median_abs_s, allclose_s in comparison_matches:
    max_abs = float(max_abs_s)
    mean_abs = float(mean_abs_s)
    median_abs = float(median_abs_s)

    if not all(
        math.isfinite(v)
        for v in (max_abs, mean_abs, median_abs)
    ):
        raise RuntimeError(
            f"Non-finite numerical parity statistic reported for {name}."
        )

    if allclose_s != "True":
        raise AssertionError(
            f"Frozen numerical comparison failed for {name}. "
            "Do NOT change atol/rtol."
        )

    comparison_stats[name] = {
        "max_abs": max_abs,
        "mean_abs": mean_abs,
        "median_abs": median_abs,
        "allclose": True,
    }

result_path_matches = re.findall(
    r"(?m)^RESULT_ZIP:\s*(.+?)\s*$",
    runner_stdout,
)

if len(result_path_matches) != 1:
    raise RuntimeError(
        "Expected exactly one RESULT_ZIP line from the frozen runner; "
        f"found {len(result_path_matches)}"
    )

runner_result_zip = Path(result_path_matches[0].strip())

print("POST prediction matches    :", f"{post_num}/{post_den}")
print("Siamese prediction matches :", f"{siamese_num}/{siamese_den}")
print("STRICT_PARITY_PASS         :", strict_pass)
print("Runner-declared result ZIP :", runner_result_zip)

if strict_pass is not True:
    raise AssertionError(
        "\nSTRICT_PARITY_PASS = FALSE.\n\n"
        "DO NOT change atol or rtol. Preserve this scientific result."
    )

if runner_result_zip != RESULT_ZIP:
    raise RuntimeError(
        "Frozen runner declared an unexpected result-ZIP path:\n"
        f"runner: {runner_result_zip}\n"
        f"locked wrapper path: {RESULT_ZIP}"
    )

if not runner_result_zip.is_file():
    raise RuntimeError(
        f"Frozen runner reported a result ZIP that does not exist: "
        f"{runner_result_zip}"
    )

print("\nAUTHORITATIVE FROZEN-RUNNER VERDICT: PASS")
print("STRICT CHECKPOINT PARITY: PASS")


# ============================================================
# ============================================================
# 9. VERIFY RESULT ZIP
# ============================================================

print("\n[9] VERIFYING EXACT RESULT ARTIFACT")

result_zip = runner_result_zip

if result_zip != RESULT_ZIP:
    raise RuntimeError(
        "Internal wrapper inconsistency: runner result path no longer "
        "matches the frozen expected result path."
    )

if not result_zip.is_file():
    raise RuntimeError(
        f"Expected result ZIP was not created: {result_zip}"
    )

with zipfile.ZipFile(result_zip, "r") as z:
    names = z.namelist()

    if not names:
        raise RuntimeError("Result ZIP is empty.")

    bad = z.testzip()

    if bad is not None:
        raise RuntimeError(
            f"Results ZIP CRC failure: {bad}"
        )

result_sha = sha256_file(result_zip)

print("\nRESULT ZIP =", result_zip)
print("RESULT ZIP FILE COUNT =", len(names))
print("RESULT ZIP BYTES =", result_zip.stat().st_size)
print("RESULT ZIP SHA256 =", result_sha)
print("RESULT ZIP CRC: PASS")
print("RUNNER STDOUT =", runner_stdout_path)

print("\nNumerical parity summaries:")
for name in sorted(comparison_stats):
    print(" ", name, "=", comparison_stats[name])

print("\n" + "=" * 70)
print("G3 CHECKPOINT-INFERENCE PARITY AUDIT COMPLETE")
print("STRICT_PARITY_PASS =", strict_pass)
print("=" * 70)


DamageACT-U G3 Frozen Checkpoint Parity

[1] ENVIRONMENT
Python      : 3.12.13
Torch       : 2.10.0+cu128
Torchvision : 0.25.0+cu128
CUDA        : 12.8
GPU         : Tesla T4
Pillow      : 11.3.0
ENVIRONMENT GATE: PASS

[2] ATTACHED INPUTS
 - /kaggle/input/datasets

[3] LOCATING FROZEN EXECUTION BUNDLE
Attached dataset roots = 3
 - /kaggle/input/datasets/rayanhossain239/damageactu-g3-parity-execution-bundle
 - /kaggle/input/datasets/rayanhossain239/damageactu-g3-frozen-archive
 - /kaggle/input/datasets/rayanhossain239/damageactu-xbd-full

Extracted-bundle signature found: /kaggle/input/datasets/rayanhossain239/damageactu-g3-parity-execution-bundle
  runner identity        = True
  runner manifest        = True
  replay lock            = True
  complete payload tree  = NO
  missing:
   - manifests/buildings/event_test_buildings_rebuilt.csv.gz
   - predictions/heldout_events/event_test_clean_expert_predictions.csv.gz

Exact preserved archive candidate(s):
 - /kaggle/input/datasets/rayanh